# NB10 Proof of stake — Chapter 10

A chapter notebook for *Blockchain for Lawyers: A Reading Through Technology and Economics*. The code is the same as in the text.

- All code here is a **toy implementation** meant to make a principle visible. Do not use it with real assets, real private keys or real transactions.
- No external API is called, and everything runs without a network. Cells that need the bundled snapshot data carry that data inside the cell.
- Standard library only (run and checked on Python 3.13). There is no install cell.
- The outputs are the editor's own run, stored as produced. Running the cells again should give the same values (random seeds are fixed).

In [ ]:
# NB10-1: stake-weighted random selection — checking "one coin, one vote" by simulation (for study, fixed seed)
import random
rng = random.Random(20261025)

stakes = {"A": 32, "B": 32, "C": 64, "D": 128, "E": 256}      # stake per validator (toy values in ETH)
total = sum(stakes.values())
names, weights = list(stakes), list(stakes.values())

N = 100_000
picks = {n: 0 for n in names}
for _ in range(N):                                             # in each slot, draw one proposer in proportion to stake
    picks[rng.choices(names, weights=weights)[0]] += 1

print(f"{'validator':>9} {'stake':>6} {'stake share':>11} {'pick share':>10}")
for n in names:
    print(f"{n:>9} {stakes[n]:>6} {stakes[n]/total:>11.3f} {picks[n]/N:>10.3f}")
print('Twice the stake, twice the picks: "one coin, one vote." If the reward is proportional to the number of picks, it is proportional to stake.')
print('What is preserved, though, is the ratio: if everyone grows at the same rate, the absolute gap grows at that rate and the ratio stays the same. On this point misconception ⑩ in Appendix B is judged "Holds in part."')

validator  stake stake share pick share
        A     32       0.062      0.062
        B     32       0.062      0.061
        C     64       0.125      0.126
        D    128       0.250      0.253
        E    256       0.500      0.498
Twice the stake, twice the picks: "one coin, one vote." If the reward is proportional to the number of picks, it is proportional to stake.
What is preserved, though, is the ratio: if everyone grows at the same rate, the absolute gap grows at that rate and the ratio stays the same. On this point misconception ⑩ in Appendix B is judged "Holds in part."


In [ ]:
# NB10-2: slashing penalties — the immediate penalty and the correlation penalty under the current specification's rules (Electra and later), keeping the specification's order of integer divisions
GWEI = 10**9
MIN_SLASHING_PENALTY_QUOTIENT_ELECTRA = 4096
WHISTLEBLOWER_REWARD_QUOTIENT_ELECTRA = 4096
PROPORTIONAL_SLASHING_MULTIPLIER_BELLATRIX = 3
EFFECTIVE_BALANCE_INCREMENT = 1 * GWEI                        # 1 ETH = 10^9 Gwei (presets/mainnet/phase0.yaml)

def immediate_penalty(effective_balance_gwei: int) -> int:     # slash_validator
    return effective_balance_gwei // MIN_SLASHING_PENALTY_QUOTIENT_ELECTRA

def correlation_penalty(effective_balance_gwei: int, slashed_sum_gwei: int, total_active_gwei: int) -> int:   # process_slashings
    increment = EFFECTIVE_BALANCE_INCREMENT
    adjusted = min(slashed_sum_gwei * PROPORTIONAL_SLASHING_MULTIPLIER_BELLATRIX, total_active_gwei)
    per_increment = adjusted // (total_active_gwei // increment)
    return (effective_balance_gwei // increment) * per_increment

eb = 32 * GWEI                                                 # one 32 ETH validator
total_active = 1_000_000 * 32 * GWEI                           # assumption: 1 million validators of 32 ETH = 32 million ETH active (a toy assumed value)
print(f"immediate penalty: {immediate_penalty(eb)/GWEI:.7f} ETH (= 32/4096), whistleblower reward: {eb // WHISTLEBLOWER_REWARD_QUOTIENT_ELECTRA / GWEI:.7f} ETH")
print(f"{'share slashed in same period':>28} {'correlation penalty (ETH)':>25} {'of balance':>10}")
for frac in [0.0001, 0.01, 0.10, 1/3, 0.5]:
    slashed_sum = int(total_active * frac)
    pen = correlation_penalty(eb, slashed_sum, total_active)
    print(f"{frac:>28.4%} {pen/GWEI:>25.4f} {pen/eb:>10.1%}")
print("Break the rules alone and the second penalty is close to 0; if 1/3 of all stake breaks them together, 3 × 1/3 = 100% — everything is lost.")

immediate penalty: 0.0078125 ETH (= 32/4096), whistleblower reward: 0.0078125 ETH
share slashed in same period correlation penalty (ETH) of balance
                     0.0100%                    0.0096       0.0%
                     1.0000%                    0.9600       3.0%
                    10.0000%                    9.6000      30.0%
                    33.3333%                   32.0000     100.0%
                    50.0000%                   32.0000     100.0%
Break the rules alone and the second penalty is close to 0; if 1/3 of all stake breaks them together, 3 × 1/3 = 100% — everything is lost.
